# mnist-demo

Companion notebook for Tutorial 9 (Section 9.4). Run it in JupyterLab on an Azure ML compute instance with the `Python 3.10 - SDK v2` kernel.

In [ ]:
# Import libraries
import tensorflow as tf
from tensorflow import keras
from keras.datasets import mnist
from keras.models import Model
from keras.layers import Input, Dense, Dropout, Flatten, Conv2D, MaxPooling2D
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load the data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Scale the pixel values to [0, 1] and add a channel dimension
X_train = X_train[..., np.newaxis] / 255.0
X_test = X_test[..., np.newaxis] / 255.0

print('Data shape:', X_train.shape)

In [ ]:
# Define the layers in the model
inputs = Input(shape=(28, 28, 1))
conv1a = Conv2D(filters=32, kernel_size=3, padding='same', activation='relu')(inputs)
conv1b = Conv2D(filters=64, kernel_size=3, padding='same', activation='relu')(conv1a)
pool1 = MaxPooling2D()(conv1b)
flat = Flatten()(pool1)
dense1 = Dense(1024, activation='relu')(flat)
dropout1 = Dropout(0.5)(dense1)
outputs = Dense(10, activation='softmax')(dropout1)

# Define the model with inputs and outputs
model = Model(inputs, outputs)

In [ ]:
# Compile the model
model.compile(optimizer="adam", loss='sparse_categorical_crossentropy', metrics=["accuracy"])

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION = "...enter your 32-digit subscription ID here..."
GROUP = "My_resource_group_1"
WORKSPACE = "Workspace_2"

# Create a client for the workspace using the subscription information
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION,
    resource_group_name=GROUP,
    workspace_name=WORKSPACE,
)

In [ ]:
import mlflow

# Track the experiment in the Azure ML workspace
mlflow_tracking_uri = ml_client.workspaces.get(WORKSPACE).mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment("demo-mnist-training")

# Automatically log the training progress
mlflow.tensorflow.autolog()

In [ ]:
# Train the model
with mlflow.start_run() as run:
    model.fit(X_train, y_train, epochs=5)

In [ ]:
# Register the model from the MLflow run
registered_model = mlflow.register_model(
    model_uri=f"runs:/{run.info.run_id}/model",
    name="mnist-tf-model",
)

print("Registered model version:", registered_model.version)

In [ ]:
# Load the registered model
loaded_model = mlflow.tensorflow.load_model(f"models:/mnist-tf-model/{registered_model.version}")

In [ ]:
# Evaluate the model
loaded_model.evaluate(X_test, y_test, verbose=0)

In [ ]:
# Show images and labels
def show_images(images, labels):
    fig, axes = plt.subplots(1, len(images), figsize=(1.5 * len(images), 2))
    for ax, image, label in zip(axes, images, labels):
        ax.imshow(image.squeeze(), cmap='gray')
        ax.set_title(label)
        ax.axis('off')
    plt.show()

# Predict the labels for several images
preds = loaded_model.predict(X_test[:10]).argmax(axis=1)
show_images(X_test[:10], preds[:10])